# Train the floor-marker reader (YOLO-pose)

Trains a small Ultralytics YOLO-**pose** model on the floor markers (four AprilTags in a red frame). For each marker it predicts the station (or `unknown_marker` when no tag is readable) and **20 keypoints**: the 4 red-frame corners and the 4 corners of each tag, also where they are covered.

Guarding against overfitting:
- nano model, strong colour/brightness augmentation, no flips (a mirrored tag is not a valid tag);
- early stopping when the validation score stops improving;
- the train vs validation loss curves are plotted (a widening gap = overfitting);
- a **held-out test set** (damage colours/shapes, floor mats and light never seen in training) is scored once, at the end.

**Steps**: Runtime > Change runtime type > **T4 GPU**, then run the cells in order. Upload `markers_v2.zip` and `markers_v2_test.zip` (made by `tools/ai/marker_dataset/pack.sh`). The last cell downloads `marker_reader_v2.zip`; put it in the workspace's `ai_data/` folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q ultralytics==8.3.40 onnx onnxslim onnxscript

In [ ]:
# Save the training to Google Drive: a disconnect then loses nothing.
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/tugger_runs'
import os; os.makedirs(RUNS, exist_ok=True)


In [ ]:
# The zips are read from Google Drive (MyDrive/tugger_data) if they are there;
# otherwise you are asked to upload them once, and they are kept in Drive.
import os, shutil
from google.colab import files
DATA = '/content/drive/MyDrive/tugger_data'
os.makedirs(DATA, exist_ok=True)
NEEDED = ['markers_v2.zip', 'markers_v2_test.zip']
missing = [z for z in NEEDED if not os.path.exists(f'{DATA}/{z}')]
if missing:
    print('Upload:', missing)
    for z in files.upload():
        shutil.move(z, f'{DATA}/{z}')
!unzip -q -o {DATA}/markers_v2.zip -d /content/
!unzip -q -o {DATA}/markers_v2_test.zip -d /content/
!ls /content

In [ ]:
import yaml
cfg = yaml.safe_load(open('/content/markers_v2/data.yaml'))
cfg['path'] = '/content/markers_v2'
yaml.safe_dump(cfg, open('/content/markers_v2/data_colab.yaml', 'w'))
test = dict(cfg, path='/content/markers_v2_test', train='images/test', val='images/test')
yaml.safe_dump(test, open('/content/markers_v2_test/data_test.yaml', 'w'))
cfg

In [ ]:
import numpy as np
np.trapz = np.trapezoid   # numpy 2 renamed it; Ultralytics 8.3.40 still calls the old name
from ultralytics import YOLO
model = YOLO('yolo11n-pose.pt')
model.train(data='/content/markers_v2/data_colab.yaml', imgsz=640, epochs=150, batch=32,
            patience=25, fliplr=0.0, flipud=0.0, degrees=0.0, mosaic=0.5, scale=0.3,
            translate=0.1, hsv_h=0.01, hsv_s=0.4, hsv_v=0.5, weight_decay=0.0005,
            project=RUNS, name='markers_pose', seed=11, deterministic=True)

In [ ]:
# ONLY if the training stopped (disconnect, restart): run cells 1, 3 (Drive) and this one;
# it continues from the last finished epoch.
import numpy as np
np.trapz = np.trapezoid
from ultralytics import YOLO
YOLO(f'{RUNS}/markers_pose/weights/last.pt').train(resume=True)


In [ ]:
# Overfitting check: training and validation losses side by side.
import pandas as pd, matplotlib.pyplot as plt
r = pd.read_csv(f'{RUNS}/markers_pose/results.csv')
r.columns = [c.strip() for c in r.columns]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, name in zip(axes, ['pose_loss', 'box_loss']):
    ax.plot(r['epoch'], r[f'train/{name}'], label='train')
    ax.plot(r['epoch'], r[f'val/{name}'], label='validation')
    ax.set_title(name); ax.set_xlabel('epoch'); ax.legend()
plt.tight_layout(); plt.savefig(f'{RUNS}/markers_pose/overfitting_check.png', dpi=150); plt.show()

In [ ]:
best = YOLO(f'{RUNS}/markers_pose/weights/best.pt')
val = best.val(data='/content/markers_v2/data_colab.yaml', imgsz=640, project=RUNS, name='val')
test = best.val(data='/content/markers_v2_test/data_test.yaml', imgsz=640, project=RUNS, name='heldout_test')
summary = {'val_pose_mAP50-95': float(val.pose.map), 'val_box_mAP50-95': float(val.box.map),
           'test_pose_mAP50-95': float(test.pose.map), 'test_box_mAP50-95': float(test.box.map)}
print(summary)
import json; json.dump(summary, open(f'{RUNS}/markers_pose/summary.json', 'w'), indent=1)
best.export(format='onnx', imgsz=640, opset=12, simplify=True)

In [ ]:
!cd {RUNS} && zip -q -r /content/marker_reader_v2.zip markers_pose val heldout_test -x '*/weights/last.pt'
from google.colab import files
files.download('/content/marker_reader_v2.zip')